### Semantic text search using embeddings

We can search through all our reviews semantically in a very efficient manner and at very low cost, by embedding our search query, and then finding the most similar reviews. The dataset is created in the [02_get_embeddings_from_dataset Notebook](02_get_embeddings_from_dataset.ipynb).

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in your directory
load_dotenv()

# Retrieve the API key from the environment
api_key = os.getenv("OPENAI_API_KEY")

# Verify if the key was loaded successfully
if api_key:
    print(f"✓ API Key loaded successfully. Starts with: {api_key[:7]}...")
else:
    print("✗ API Key not found. Please check your .env file.")

✓ API Key loaded successfully. Starts with: sk-proj...


In [3]:
import pandas as pd
import numpy as np
from ast import literal_eval

from openai import OpenAI
import pandas as pd

client = OpenAI(max_retries=5)

In [4]:
datafile_path = "data/fine_food_reviews_with_embeddings_1k.csv"

df = pd.read_csv(datafile_path)
df.head

<bound method NDFrame.head of      Unnamed: 0   ProductId          UserId  Score  \
0             0  B003XPF9BO  A3R7JR3FMEBXQB      5   
1           297  B003VXHGPK  A21VWSCGW7UUAR      4   
2           296  B008JKTTUA  A34XBAIFT02B60      1   
3           295  B000LKTTTW  A14MQ40CCU8B13      5   
4           294  B001D09KAM  A34XBAIFT02B60      1   
..          ...         ...             ...    ...   
995         623  B0000CFXYA  A3GS4GWPIBV0NT      1   
996         624  B0001BH5YM   A1BZ3HMAKK0NC      5   
997         625  B0009ET7TC  A2FSDQY5AI6TNX      5   
998         619  B007PA32L2  A15FF2P7RPKH6G      5   
999         999  B001EQ5GEO  A3VYU0VO6DYV6I      5   

                                               Summary  \
0    where does one  start...and stop... with a tre...   
1                     Good, but not Wolfgang Puck good   
2    Should advertise coconut as an ingredient more...   
3                                     Best tomato soup   
4    Should advertise coconut a

In [5]:
df["embedding"] = df.embedding.apply(literal_eval).apply(np.array)
df.head

<bound method NDFrame.head of      Unnamed: 0   ProductId          UserId  Score  \
0             0  B003XPF9BO  A3R7JR3FMEBXQB      5   
1           297  B003VXHGPK  A21VWSCGW7UUAR      4   
2           296  B008JKTTUA  A34XBAIFT02B60      1   
3           295  B000LKTTTW  A14MQ40CCU8B13      5   
4           294  B001D09KAM  A34XBAIFT02B60      1   
..          ...         ...             ...    ...   
995         623  B0000CFXYA  A3GS4GWPIBV0NT      1   
996         624  B0001BH5YM   A1BZ3HMAKK0NC      5   
997         625  B0009ET7TC  A2FSDQY5AI6TNX      5   
998         619  B007PA32L2  A15FF2P7RPKH6G      5   
999         999  B001EQ5GEO  A3VYU0VO6DYV6I      5   

                                               Summary  \
0    where does one  start...and stop... with a tre...   
1                     Good, but not Wolfgang Puck good   
2    Should advertise coconut as an ingredient more...   
3                                     Best tomato soup   
4    Should advertise coconut a

Next, we compare the cosine similarity of the embeddings of a query and the documents, and show top_n best matches.

In [6]:
from typing import List, Optional
#def get_embedding(text: str, model="text-embedding-3-small", **kwargs) -> List[float]:
def get_embedding(text, model="text-embedding-3-small", **kwargs) -> List[float]:
    # replace newlines, which can negatively affect performance.
    text = text.replace("\n", " ")

    response = client.embeddings.create(input=[text], model=model, **kwargs)

    return response.data[0].embedding


In [7]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [8]:
#from utils.embeddings_utils import get_embedding, cosine_similarity

# search through the reviews for a specific product
def search_reviews(df, product_description, n=3, pprint=True):
    product_embedding = get_embedding(
        product_description,
        model="text-embedding-3-small"
    )
    df["similarity"] = df.embedding.apply(lambda x: cosine_similarity(x, product_embedding))

    results = (
        df.sort_values("similarity", ascending=False)
        .head(n)
        .combined.str.replace("Title: ", "")
        .str.replace("; Content:", ": ")
    )
    if pprint:
        for r in results:
            print(r[:200])
            print()
    return results


results = search_reviews(df, "delicious beans", n=3)


Delicious!:  I enjoy this white beans seasoning, it gives a rich flavor to the beans I just love it, my mother in law didn't know about this Zatarain's brand and now she is traying different seasoning

Fantastic Instant Refried beans:  Fantastic Instant Refried Beans have been a staple for my family now for nearly 20 years.  All 7 of us love it and my grown kids are passing on the tradition.

Delicious:  While there may be better coffee beans available, this is my first purchase and my first time grinding my own beans.  I read several reviews before purchasing this brand, and am extremely 



In [9]:
results = search_reviews(df, "whole wheat pasta", n=3)


Tasty and Quick Pasta:  Barilla Whole Grain Fusilli with Vegetable Marinara is tasty and has an excellent chunky vegetable marinara.  I just wish there was more of it.  If you aren't starving or on a 

sooo good:  tastes so good. Worth the money. My boyfriend hates wheat pasta and LOVES this. cooks fast tastes great.I love this brand and started buying more of their pastas. Bulk is best.

Bland and vaguely gamy tasting, skip this one:  As far as prepared dinner kits go, "Barilla Whole Grain Mezze Penne with Tomato and Basil Sauce" just did not do it for me...and this is coming from a p



We can search through these reviews easily. To speed up computation, we can use a special algorithm, aimed at faster search through embeddings.

In [10]:
results = search_reviews(df, "bad delivery", n=1)


great product, poor delivery:  The coffee is excellent and I am a repeat buyer.  Problem this time was with the UPS delivery.  They left the box in front of my garage door in the middle of the drivewa



As we can see, this can immediately deliver a lot of value. In this example we show being able to quickly find the examples of delivery failures.

In [11]:
results = search_reviews(df, "spoilt", n=1)


Disappointed:  The metal cover has severely disformed. And most of the cookies inside have been crushed into small pieces. Shopping experience is awful. I'll never buy it online again.



In [12]:
results = search_reviews(df, "pet food", n=2)


Good food:  The only dry food my queen cat will eat. Helps prevent hair balls. Good packaging. Arrives promptly. Recommended by a friend who sells pet food.

Great food!:  I wanted a food for a a dog with skin problems. His skin greatly improved with the switch, though he still itches some.  He loves the food. No recalls, American made with American ingred

